###### Notebook 1 — 03_Load_Silver_to_Gold_Augment.py

In [0]:
# ── Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# Load shared pipeline config
%run ../../config/pipeline_config.py

In [0]:
%run "../../utils/metrics_query"


In [0]:
mq = MetricsQuery(spark)
print(":white_check_mark: MetricsQuery loaded")

In [0]:
%run "../../utils/audit_logger"

In [0]:
# CDC validator — commented out per requirement
# %run ../../utils/cdc_validator

In [0]:
from pyspark.sql.functions import col, lit, lower, when, current_timestamp, coalesce, split
from delta.tables import DeltaTable
import time

# ===================================================================================
# CONFIGURATION
# ===================================================================================
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").lower()

# Object-specific constants
source_system  = "augment"
object_name    = "gold_augment"
load_type      = "incremental"  # hardcoded — change to "historical" if full backfill needed

# ===================================================================================
# FUTURE ENHANCEMENT — Multi-Customer Loop (uncomment when ready):
#
# customer_ids_df = spark.sql("""
#     SELECT DISTINCT customer_id
#     FROM ingestion_metadata.customers
#     WHERE is_active = true
# """)
# customer_ids = [row.customer_id for row in customer_ids_df.collect()]
# for customer_id in customer_ids:
#     pass
# ===================================================================================

SILVER_CATALOG     = "hgi"
SILVER_SCHEMA      = "silver"
GOLD_CATALOG       = "hgi"
GOLD_SCHEMA        = "gold"
ENRICHMENT_CATALOG = "hgi"
ENRICHMENT_SCHEMA  = "enrichment"
STALE_DAYS         = 180

FREE_EMAIL_DOMAINS = [
    'gmail.com', 'yahoo.com', 'hotmail.com',
    'aol.com', 'outlook.com', 'icloud.com'
]

spark.conf.set("spark.sql.shuffle.partitions", 4)

print("=" * 70)
print("03_Load_Silver_to_Gold_Augment")
print(f"  Customer : {customer_id}")
print(f"  Load Type: {load_type}")
print("=" * 70)

In [0]:
import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "03_Load_Silver_to_Gold_Augment",
    task_key       = "03_Load_Silver_to_Gold_Augment",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()
print("✅ PipelineMetrics & AuditLogger initialized")

In [0]:
# ===================================================================================
# CDF STATUS CHECK — Silver 2a Tables
# Checks if Change Data Feed is enabled and prints updated record counts
# ===================================================================================
from pyspark.sql.functions import col as _col

def check_cdf_status(table_full_name):
    """Check if CDF is enabled on a Delta table."""
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {table_full_name}") \
            .filter("key = 'delta.enableChangeDataFeed'").collect()
        return len(props) > 0 and props[0]["value"].lower() == "true"
    except Exception:
        return False

def get_cdf_updated_counts(table_full_name):
    """Get count of updated/inserted records from latest CDF version."""
    try:
        history = spark.sql(f"DESCRIBE HISTORY {table_full_name} LIMIT 2").collect()
        if not history:
            return None, 0
        latest_version = history[0]["version"]
        changes_df = spark.read.format("delta") \
            .option("readChangeFeed", "true") \
            .option("startingVersion", latest_version) \
            .table(table_full_name)
        update_count = changes_df.filter(
            _col("_change_type").isin("update_postimage", "insert")
        ).count()
        return latest_version, update_count
    except Exception as e:
        return None, str(e)[:100]

silver_2a_tables = [
    "accounts", "contacts", "opportunities", "tasks",
    "campaigns", "campaign_members", "users", "events"
]

print("=" * 70)
print("CDF STATUS — Silver 2a Tables")
print("=" * 70)
for tbl_name in silver_2a_tables:
    full_name = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.{tbl_name}"
    cdf_on = check_cdf_status(full_name)
    if cdf_on:
        version, count = get_cdf_updated_counts(full_name)
        if version is not None:
            print(f"  \u2705 {tbl_name}: CDF enabled | version {version} | {count} updated records")
        else:
            print(f"  \u2705 {tbl_name}: CDF enabled | no version history yet")
    else:
        print(f"  \u26a0\ufe0f  {tbl_name}: CDF NOT enabled")

# ===================================================================================
# GOLD TABLE INITIALIZATION
# ===================================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")

def initialize_gold_tables():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.persons (
            mk_email              STRING    COMMENT 'Lookup key lowercase email',
            mk_domain             STRING,
            mk_status             STRING,
            mk_timestamp          TIMESTAMP,
            name__fullname        STRING,
            name__givenname       STRING,
            name__familyname      STRING,
            employment__name      STRING,
            employment__domain    STRING,
            employment__role      STRING,
            employment__seniority STRING,
            employment__title     STRING,
            geo__city             STRING,
            geo__country          STRING,
            geo__state            STRING,
            is_student            BOOLEAN,
            is_spam               BOOLEAN,
            linkedin__handle      STRING,
            twitter__handle       STRING,
            updated_at            TIMESTAMP
        ) USING DELTA CLUSTER BY (mk_email)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.companies (
            mk_domain               STRING    COMMENT 'Lookup key lowercase domain',
            mk_status               STRING,
            mk_timestamp            TIMESTAMP,
            name                    STRING,
            description             STRING,
            category__industry      STRING,
            category__sector        STRING,
            category__industrygroup STRING,
            category__subindustry   STRING,
            metrics__employees      LONG,
            metrics__employeesrange STRING,
            metrics__raised         LONG,
            metrics__marketcap      STRING,
            geo__city               STRING,
            geo__country            STRING,
            geo__state              STRING,
            tech                    STRING,
            tags                    STRING,
            founded_year            LONG,
            emailprovider           BOOLEAN,
            site_url                STRING,
            alexa__globalrank       LONG,
            is_personal             BOOLEAN,
            updated_at              TIMESTAMP
        ) USING DELTA CLUSTER BY (mk_domain)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations (
            contact_id                        STRING,
            tenant                           LONG,
            email                            STRING,
            domain                           STRING,
            person__mk_email                 STRING,
            person__mk_domain                STRING,
            person__mk_status                STRING,
            person__mk_timestamp             TIMESTAMP,
            person__name__fullname           STRING,
            person__name__givenname          STRING,
            person__name__familyname         STRING,
            person__employment__name         STRING,
            person__employment__domain       STRING,
            person__employment__role         STRING,
            person__employment__seniority    STRING,
            person__employment__title        STRING,
            person__geo__city                STRING,
            person__geo__country             STRING,
            person__geo__state               STRING,
            person__is_student               BOOLEAN,
            person__is_spam                  BOOLEAN,
            person__linkedin__handle         STRING,
            person__twitter__handle          STRING,
            company__mk_domain               STRING,
            company__mk_status               STRING,
            company__mk_timestamp            TIMESTAMP,
            company__name                    STRING,
            company__description             STRING,
            company__category__industry      STRING,
            company__category__sector        STRING,
            company__category__industrygroup STRING,
            company__category__subindustry   STRING,
            company__metrics__employees      LONG,
            company__metrics__employeesrange STRING,
            company__metrics__raised         LONG,
            company__metrics__marketcap      STRING,
            company__geo__city               STRING,
            company__geo__country            STRING,
            company__geo__state              STRING,
            company__tech                    STRING,
            company__tags                    STRING,
            company__founded_year            LONG,
            company__emailprovider           BOOLEAN,
            company__site_url                STRING,
            company__alexa__globalrank       LONG,
            company__is_personal             BOOLEAN,
            updated_at                       TIMESTAMP
        ) USING DELTA CLUSTER BY (contact_id)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

initialize_gold_tables()
print("\u2705 Gold tables initialized.")

# ===================================================================================
# WATERMARK HELPERS
# ===================================================================================
def get_watermark(src_sys, obj_name):
    try:
        result = spark.sql(f"""
            SELECT last_processed_timestamp
            FROM ingestion_metadata.watermarks
            WHERE source_system = '{src_sys}' AND object_name = '{obj_name}'
        """).collect()
        if result:
            return result[0]["last_processed_timestamp"]
    except Exception:
        pass
    return None

def update_watermark(src_sys, obj_name):
    spark.sql(f"""
        MERGE INTO ingestion_metadata.watermarks AS target
        USING (SELECT '{src_sys}' AS source_system, '{obj_name}' AS object_name,
                      current_timestamp() AS last_processed_timestamp) AS source
        ON target.source_system = source.source_system
        AND target.object_name  = source.object_name
        WHEN MATCHED THEN UPDATE SET
            target.last_processed_timestamp = source.last_processed_timestamp
        WHEN NOT MATCHED THEN INSERT
            (source_system, object_name, last_processed_timestamp)
            VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
    """)
    print(f"  Watermark updated: source_system='{src_sys}', object_name='{obj_name}'")

In [0]:
# ===================================================================================
# MAIN PIPELINE
# ===================================================================================
start_time               = time.time()
email_count              = 0
domain_count             = 0
persons_enriched_count   = 0
companies_enriched_count = 0
compute_count            = 0
total_rows               = 0

try:

    # -----------------------------------------------------------------------
    # STEP 0A — EMAIL CANDIDATES
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 0A — EMAIL CANDIDATES DATAFRAME")
    print("="*70)

    if load_type == "historical":
        df_email_candidates = spark.sql(f"""
            SELECT DISTINCT lower(email) AS email
            FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all
            WHERE email IS NOT NULL
        """)
    else:
        df_email_candidates = spark.sql(f"""
            WITH candidates AS (
                SELECT DISTINCT lower(ca.email) AS email, 1 AS priority
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts uc ON ca.id = uc.id
                WHERE ca.email IS NOT NULL
                UNION
                SELECT DISTINCT lower(ca.email) AS email, 2 AS priority
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.events e ON ca.id = e.contact_id
                WHERE ca.email IS NOT NULL
                  AND e.event_timestamp >= date_add(current_date(), -30)
                UNION
                SELECT DISTINCT lower(ca.email) AS email, 3 AS priority
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.events e ON ca.id = e.contact_id
                WHERE ca.email IS NOT NULL
                  AND e.event_timestamp >= date_add(current_date(), -7)
            ),
            ranked AS (
                SELECT email, MIN(priority) AS priority
                FROM candidates GROUP BY email
            )
            SELECT r.email
            FROM ranked r
            LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p ON r.email = lower(p.mk_email)
            WHERE p.mk_email IS NULL
               OR p.mk_status IN ('error', 'not_found')
               OR (p.mk_timestamp IS NOT NULL AND p.mk_timestamp < date_add(current_date(), -{STALE_DAYS}))
            ORDER BY r.priority ASC
        """)

    email_count = df_email_candidates.count()
    print(f"  Email candidates: {email_count}")
    df_email_candidates.show(20, truncate=False)

    # -----------------------------------------------------------------------
    # STEP 0B — DOMAIN CANDIDATES
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 0B — DOMAIN CANDIDATES DATAFRAME")
    print("="*70)

    if load_type == "historical":
        df_domain_candidates = spark.sql(f"""
            SELECT DISTINCT lower(domain) AS domain
            FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all
            WHERE domain IS NOT NULL
        """)
    else:
        df_domain_candidates = spark.sql(f"""
            SELECT DISTINCT lower(a.domain) AS domain
            FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all a
            LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.companies c ON lower(a.domain) = lower(c.mk_domain)
            WHERE a.domain IS NOT NULL
              AND (c.mk_domain IS NULL
                   OR c.mk_status IN ('error', 'not_found')
                   OR (c.mk_timestamp IS NOT NULL AND c.mk_timestamp < date_add(current_date(), -{STALE_DAYS})))
        """)

    domain_count = df_domain_candidates.count()
    print(f"  Domain candidates: {domain_count}")
    df_domain_candidates.show(20, truncate=False)

    # -----------------------------------------------------------------------
    # STEP 1 — EMAIL ENRICHMENT
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 1 — EMAIL ENRICHMENT")
    print("="*70)

    df_person_profiles  = spark.table(f"{ENRICHMENT_CATALOG}.{ENRICHMENT_SCHEMA}.person_profiles")
    df_persons_enriched = df_email_candidates.alias("cand").join(
        df_person_profiles.alias("p"),
        lower(col("cand.email")) == lower(col("p.mk_email")), "left"
    ).select(
        coalesce(lower(col("p.mk_email")), lower(col("cand.email"))).alias("mk_email"),
        when(col("p.mk_domain").isNotNull(), col("p.mk_domain"))
            .otherwise(when(col("cand.email").contains("@"),
                            split(lower(col("cand.email")), "@").getItem(1))
                       .otherwise(lit(None))).alias("mk_domain"),
        when(col("p.mk_status").isNotNull(), col("p.mk_status")).otherwise(lit("not_found")).alias("mk_status"),
        when(col("p.mk_timestamp").isNotNull(), col("p.mk_timestamp")).otherwise(current_timestamp()).alias("mk_timestamp"),
        col("p.name__fullname"), col("p.name__givenname"), col("p.name__familyname"),
        col("p.employment__name"), col("p.employment__domain"), col("p.employment__role"),
        col("p.employment__seniority"), col("p.employment__title"),
        col("p.geo__city"), col("p.geo__country"), col("p.geo__state"),
        col("p.is_student"), col("p.is_spam"),
        col("p.linkedin__handle"), col("p.twitter__handle"),
        current_timestamp().alias("updated_at")
    )

    persons_enriched_count = df_persons_enriched.count()
    print(f"  Persons enriched: {persons_enriched_count}")
    df_persons_enriched.show(20, truncate=True)

    DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.persons").alias("target").merge(
        df_persons_enriched.alias("source"),
        "lower(target.mk_email) = lower(source.mk_email)"
    ).whenMatchedUpdate(set={
        "mk_domain":"source.mk_domain","mk_status":"source.mk_status",
        "mk_timestamp":"source.mk_timestamp","name__fullname":"source.name__fullname",
        "name__givenname":"source.name__givenname","name__familyname":"source.name__familyname",
        "employment__name":"source.employment__name","employment__domain":"source.employment__domain",
        "employment__role":"source.employment__role","employment__seniority":"source.employment__seniority",
        "employment__title":"source.employment__title","geo__city":"source.geo__city",
        "geo__country":"source.geo__country","geo__state":"source.geo__state",
        "is_student":"source.is_student","is_spam":"source.is_spam",
        "linkedin__handle":"source.linkedin__handle","twitter__handle":"source.twitter__handle",
        "updated_at":"source.updated_at"
    }).whenNotMatchedInsertAll().execute()

    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.persons").show(20, truncate=True)
    print(f"STEP 1 COMPLETE — {persons_enriched_count} persons written to hgi.gold.persons")
    update_watermark("augment", "persons")

    # -----------------------------------------------------------------------
    # STEP 2 — DOMAIN ENRICHMENT
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 2 — DOMAIN ENRICHMENT")
    print("="*70)

    df_company_profiles   = spark.table(f"{ENRICHMENT_CATALOG}.{ENRICHMENT_SCHEMA}.company_profiles")
    df_companies_enriched = df_domain_candidates.alias("cand").join(
        df_company_profiles.alias("c"),
        lower(col("cand.domain")) == lower(col("c.mk_domain")), "left"
    ).select(
        coalesce(lower(col("c.mk_domain")), lower(col("cand.domain"))).alias("mk_domain"),
        when(col("c.mk_status").isNotNull(), col("c.mk_status")).otherwise(lit("not_found")).alias("mk_status"),
        current_timestamp().alias("mk_timestamp"),
        col("c.name"), col("c.description"),
        col("c.category__industry"), col("c.category__sector"),
        col("c.category__industrygroup"), col("c.category__subindustry"),
        col("c.metrics__employees"), col("c.metrics__employeesrange"),
        col("c.metrics__raised"), col("c.metrics__marketcap"),
        col("c.geo__city"), col("c.geo__country"), col("c.geo__state"),
        col("c.tech"), col("c.tags"), col("c.founded_year"),
        col("c.emailprovider"), col("c.site_url"), col("c.alexa__globalrank"),
        when(lower(col("cand.domain")).isin(FREE_EMAIL_DOMAINS), lit(True))
            .otherwise(lit(False)).alias("is_personal"),
        current_timestamp().alias("updated_at")
    )

    companies_enriched_count = df_companies_enriched.count()
    print(f"  Companies enriched: {companies_enriched_count}")
    df_companies_enriched.show(20, truncate=True)

    DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.companies").alias("target").merge(
        df_companies_enriched.alias("source"),
        "lower(target.mk_domain) = lower(source.mk_domain)"
    ).whenMatchedUpdate(set={
        "mk_status":"source.mk_status","mk_timestamp":"source.mk_timestamp",
        "name":"source.name","description":"source.description",
        "category__industry":"source.category__industry","category__sector":"source.category__sector",
        "category__industrygroup":"source.category__industrygroup",
        "category__subindustry":"source.category__subindustry",
        "metrics__employees":"source.metrics__employees",
        "metrics__employeesrange":"source.metrics__employeesrange",
        "metrics__raised":"source.metrics__raised","metrics__marketcap":"source.metrics__marketcap",
        "geo__city":"source.geo__city","geo__country":"source.geo__country",
        "geo__state":"source.geo__state","tech":"source.tech","tags":"source.tags",
        "founded_year":"source.founded_year","emailprovider":"source.emailprovider",
        "site_url":"source.site_url","alexa__globalrank":"source.alexa__globalrank",
        "is_personal":"source.is_personal","updated_at":"source.updated_at"
    }).whenNotMatchedInsertAll().execute()

    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.companies").show(20, truncate=True)
    print(f"STEP 2 COMPLETE — {companies_enriched_count} companies written to hgi.gold.companies")
    update_watermark("augment", "companies")

    # -----------------------------------------------------------------------
    # STEP 3 — COMPUTE (WIDE JOIN) ENHANCED
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 3 — COMPUTE (WIDE JOIN) ENHANCED")
    print("="*70)

    df_contacts_computations = spark.sql(f"""
        SELECT
            ca.id                                AS contact_id,
            ca.tenant, ca.email, ca.domain,
            p.mk_email                           AS person__mk_email,
            p.mk_domain                          AS person__mk_domain,
            p.mk_status                          AS person__mk_status,
            p.mk_timestamp                       AS person__mk_timestamp,
            p.name__fullname                     AS person__name__fullname,
            p.name__givenname                    AS person__name__givenname,
            p.name__familyname                   AS person__name__familyname,
            p.employment__name                   AS person__employment__name,
            p.employment__domain                 AS person__employment__domain,
            p.employment__role                   AS person__employment__role,
            p.employment__seniority              AS person__employment__seniority,
            p.employment__title                  AS person__employment__title,
            p.geo__city                          AS person__geo__city,
            p.geo__country                       AS person__geo__country,
            p.geo__state                         AS person__geo__state,
            p.is_student                         AS person__is_student,
            p.is_spam                            AS person__is_spam,
            p.linkedin__handle                   AS person__linkedin__handle,
            p.twitter__handle                    AS person__twitter__handle,
            c.mk_domain                          AS company__mk_domain,
            c.mk_status                          AS company__mk_status,
            c.mk_timestamp                       AS company__mk_timestamp,
            c.name                               AS company__name,
            c.description                        AS company__description,
            c.category__industry                 AS company__category__industry,
            c.category__sector                   AS company__category__sector,
            c.category__industrygroup            AS company__category__industrygroup,
            c.category__subindustry              AS company__category__subindustry,
            c.metrics__employees                 AS company__metrics__employees,
            c.metrics__employeesrange            AS company__metrics__employeesrange,
            c.metrics__raised                    AS company__metrics__raised,
            c.metrics__marketcap                 AS company__metrics__marketcap,
            c.geo__city                          AS company__geo__city,
            c.geo__country                       AS company__geo__country,
            c.geo__state                         AS company__geo__state,
            c.tech                               AS company__tech,
            c.tags                               AS company__tags,
            c.founded_year                       AS company__founded_year,
            c.emailprovider                      AS company__emailprovider,
            c.site_url                           AS company__site_url,
            c.alexa__globalrank                  AS company__alexa__globalrank,
            c.is_personal                        AS company__is_personal,
            current_timestamp()                  AS updated_at
        FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
        LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p
            ON lower(ca.email) = lower(p.mk_email)
        LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.companies c
            ON lower(ca.domain) = lower(c.mk_domain)
    """)

    compute_count = df_contacts_computations.count()
    print(f"  contacts_computations rows: {compute_count}")
    df_contacts_computations.show(20, truncate=True)

    DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").alias("target").merge(
        df_contacts_computations.alias("source"),
        "target.contact_id = source.contact_id AND target.tenant = source.tenant"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").show(20, truncate=True)
    print(f"STEP 3 COMPLETE — {compute_count} rows written to hgi.gold.contacts_computations")

    # -----------------------------------------------------------------------
    # CDF ENABLEMENT on gold tables
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("CDF ENABLEMENT — Enabling Change Data Feed on gold tables")
    print("="*70)

    try:
        spark.sql(f"ALTER TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.persons SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
        spark.sql(f"ALTER TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.companies SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
        spark.sql(f"ALTER TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
        print("  CDF enabled: persons, companies, contacts_computations")
    except Exception as cdf_e:
        print(f"  WARNING: CDF enablement failed: {str(cdf_e)}")

except Exception as e:
    print(f"\n\u274c Pipeline failed: {e}")
    log_audit(
        job_name       = "03_Load_Silver_to_Gold_Augment",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "gold",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
        rows_processed = total_rows,
        duration_ms    = int((time.time() - start_time) * 1000),
        email_on_failure = "pratibha.j.kumari@v4c.ai"
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# ===================================================================================
# FINAL METRICS SAVE + SUMMARY
# ===================================================================================
try:
    end_time    = time.time()
    duration_ms = round((end_time - start_time) * 1000, 2)

    persons_rows      = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons").collect()[0]["cnt"]
    companies_rows    = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies").collect()[0]["cnt"]
    computations_rows = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").collect()[0]["cnt"]
    total_rows = persons_rows + companies_rows + computations_rows

    # Save to PipelineMetrics (same as bronze/silver pattern)
    pm.save(status="SUCCESS", rows_processed=total_rows)

    # Save to MetricsQuery stats
    mq.save_stats(days=30, rows_processed=total_rows)

    print(f"""
=======================================================================
  AUGMENT LAYER COMPLETE
  Customer : {customer_id}
  Status   : SUCCESS
=======================================================================
  STEP 0A : {email_count} email candidates
  STEP 0B : {domain_count} domain candidates
  STEP 1  : {persons_enriched_count} persons  -> hgi.gold.persons
  STEP 2  : {companies_enriched_count} companies -> hgi.gold.companies
  STEP 3  : {compute_count} rows   -> hgi.gold.contacts_computations
  Duration: {duration_ms} ms
=======================================================================
  hgi.gold.persons              : {persons_rows:,} total rows
  hgi.gold.companies            : {companies_rows:,} total rows
  hgi.gold.contacts_computations: {computations_rows:,} total rows
  Pipeline metrics saved        : {total_rows:,} total rows
=======================================================================
""")
except Exception as e:
    print(f"\u274c Metrics save failed: {e}")
    log_audit(
        job_name       = "03_Load_Silver_to_Gold_Augment",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "gold",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
'''# Check 1: How many contacts exist?
spark.sql("SELECT COUNT(*) FROM hgi.silver.contacts_all WHERE email IS NOT NULL").show()

# Check 2: How many contacts hit Priority 1?
spark.sql("""
    SELECT COUNT(*) FROM hgi.silver.contacts_all ca
    JOIN hgi.silver.contacts uc ON ca.id = uc.id
    WHERE ca.email IS NOT NULL
      AND uc.data_timestamp >= date_add(current_date(), -7)
""").show()

# Check 3: How many events exist with signup/conversion?
spark.sql("""
    SELECT meta_event, COUNT(*) as cnt
    GROUP BY meta_event
""").show()

# Check 4: How many domains in accounts_all?
spark.sql("SELECT COUNT(*) FROM hgi.silver.accounts_all WHERE domain IS NOT NULL").show()'''